# pycrdt_store smoke test

Persistence layer used by pycrdt-websocket 0.16+. The `pycrdt-store` distribution installs the `pycrdt.store` module.

In [ ]:
# Example 1: the public API surface of pycrdt.store (the module pycrdt-store installs).
import importlib.metadata

import pycrdt.store

public_names = [n for n in dir(pycrdt.store) if not n.startswith('_')]
print(f"pycrdt-store {importlib.metadata.version('pycrdt-store')}: {len(public_names)} public names")
for n in public_names:
    print(f"  {n}: {type(getattr(pycrdt.store, n)).__name__}")

for name in ('BaseYStore', 'FileYStore', 'TempFileYStore', 'SQLiteYStore', 'YDocNotFound'):
    assert name in public_names, f"pycrdt.store is missing {name}"

In [ ]:
# Example 2: the concrete stores implement the abstract BaseYStore persistence interface.
import inspect

from pycrdt.store import BaseYStore, FileYStore, SQLiteYStore, TempFileYStore

assert inspect.isabstract(BaseYStore)
for cls in (FileYStore, TempFileYStore, SQLiteYStore):
    assert issubclass(cls, BaseYStore) and not inspect.isabstract(cls), cls
    assert inspect.iscoroutinefunction(cls.write), f"{cls.__name__}.write is not async"
    assert inspect.isasyncgenfunction(cls.read), f"{cls.__name__}.read is not an async iterator"
    print(f"  {cls.__name__} (mro depth {len(cls.__mro__)})")

In [ ]:
# Example 3: a FileYStore keeps its file path and touches nothing on disk until the first write.
import tempfile
from pathlib import Path

from pycrdt.store import FileYStore

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'ystore.y'
    store = FileYStore(str(path))
    print(f"FileYStore: {store!r}")
    print(f"  path attribute: {store.path}")
    assert store.path == str(path)
    assert not path.exists()

In [ ]:
# Example 4: write a Y update through a FileYStore and read it back into a fresh Doc.
import tempfile
from pathlib import Path

from pycrdt import Doc, Text
from pycrdt.store import FileYStore

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'doc.y'
    doc = Doc()
    doc['text'] = text = Text()
    text += 'hello'
    update = doc.get_update()
    async with FileYStore(str(path)) as store:
        await store.write(update)
        restored = Doc()
        await store.apply_updates(restored)
    restored_text = str(restored.get('text', type=Text))
    print(f"Wrote {len(update)} bytes to {path.name}; read back {restored_text!r}")
    assert path.exists() and path.stat().st_size > 0
    assert restored_text == 'hello', restored_text

In [ ]:
# Example 5: the runtime dependencies are importable and version metadata is reachable.
import importlib.metadata

import anyio
import pycrdt
import pycrdt.store
import sqlite_anyio

for dist, mod in (('pycrdt-store', pycrdt.store), ('pycrdt', pycrdt), ('anyio', anyio), ('sqlite-anyio', sqlite_anyio)):
    print(f"  {dist}: {importlib.metadata.version(dist)} (module {mod.__name__})")
assert importlib.metadata.version('pycrdt-store')